# Pinterest Scraper - Testing Notebook

Test the Pinterest scraper implementation:
1. **PinterestScraper** - URL-based extraction (posts, profiles)
2. **PinterestSearchScraper** - Parameter-based discovery (by keyword, by profile URL)

---

## Setup - Use Local Development Version

In [1]:
import os
import sys
from pathlib import Path

# Add local src to path (use development version, not installed)
project_root = Path.cwd().parent.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Using source from: {src_path}")

# Load environment variables
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

# Get API token
API_TOKEN = os.getenv("BRIGHTDATA_API_TOKEN")
if not API_TOKEN:
    raise ValueError("BRIGHTDATA_API_TOKEN not found in environment")

print(f"API Token: {API_TOKEN[:10]}...{API_TOKEN[-4:]}")
print("Setup complete!")

Using source from: /Users/ns/Desktop/projects/sdk-python/src
API Token: 859a9772-4...4362
Setup complete!


## Import Pinterest Scrapers

In [2]:
from brightdata import BrightDataClient

# Verify we're using local version
import brightdata
print(f"brightdata module location: {brightdata.__file__}")

# Initialize client
client = BrightDataClient(token=API_TOKEN)

# Verify Pinterest scraper is accessible
print(f"\nPinterestScraper: {type(client.scrape.pinterest).__name__}")
print(f"PinterestSearchScraper: {type(client.search.pinterest).__name__}")

# Check for scraper methods
print("\nScraper methods (URL-based):")
print([m for m in dir(client.scrape.pinterest) if not m.startswith('_') and callable(getattr(client.scrape.pinterest, m))])

print("\nSearch scraper methods (Discovery):")
print([m for m in dir(client.search.pinterest) if not m.startswith('_') and callable(getattr(client.search.pinterest, m))])

brightdata module location: /Users/ns/Desktop/projects/sdk-python/src/brightdata/__init__.py

PinterestScraper: PinterestScraper
PinterestSearchScraper: PinterestSearchScraper

Scraper methods (URL-based):
['normalize_result', 'posts', 'posts_fetch', 'posts_fetch_sync', 'posts_status', 'posts_status_sync', 'posts_sync', 'posts_trigger', 'posts_trigger_sync', 'profiles', 'profiles_fetch', 'profiles_fetch_sync', 'profiles_status', 'profiles_status_sync', 'profiles_sync', 'profiles_trigger', 'profiles_trigger_sync', 'scrape', 'scrape_async']

Search scraper methods (Discovery):
['posts_by_keyword', 'posts_by_keyword_sync', 'posts_by_profile', 'posts_by_profile_sync', 'profiles', 'profiles_sync']


---
# Part 1: PinterestScraper (URL-based Extraction)

Collect structured data from Pinterest URLs.

## 1.1 Posts - Collect pin data by URL

In [4]:
# Collect pin data by URL
PIN_URLS = [
    "https://www.pinterest.com/pin/3166662230556591/",
    "https://www.pinterest.com/pin/99360735523397542/",
]

print(f"Scraping {len(PIN_URLS)} pins...")
for url in PIN_URLS:
    print(f"  - {url}")
print("This may take 1-3 minutes...\n")

async with client.scrape.pinterest.engine:
    results = await client.scrape.pinterest.posts(url=PIN_URLS, timeout=240)

if isinstance(results, list):
    for i, result in enumerate(results):
        print(f"\n--- Pin {i+1} ---")
        print(f"Success: {result.success}")
        print(f"Cost: ${result.cost:.4f}" if result.cost else "Cost: N/A")
        if result.success and result.data:
            data = result.data
            if isinstance(data, dict) and 'error' not in data:
                print(f"Title: {data.get('title', 'N/A')}")
                print(f"Content: {str(data.get('content', 'N/A'))[:100]}")
                print(f"Image/Video URL: {data.get('image_video_url', 'N/A')}")
                print(f"Likes: {data.get('likes', 'N/A')}")
                print(f"Comments: {data.get('comments_num', 'N/A')}")
                print(f"Post Type: {data.get('post_type', 'N/A')}")
                print(f"User: {data.get('user_name', 'N/A')}")
                print(f"Date: {data.get('date_posted', 'N/A')}")
            elif isinstance(data, dict) and 'error' in data:
                print(f"Error: {data.get('error')} (code: {data.get('error_code')})")
else:
    print(f"Success: {results.success}")
    print(f"Status: {results.status}")
    if results.success and results.data:
        print(f"Data: {str(results.data)[:300]}")
    else:
        print(f"Error: {results.error}")

Scraping 2 pins...
  - https://www.pinterest.com/pin/3166662230556591/
  - https://www.pinterest.com/pin/99360735523397542/
This may take 1-3 minutes...


--- Pin 1 ---
Success: True
Cost: $0.0010
Title: 12 Bombshell Hair Color Ideas To Try This Summer | Ecemella
Content: Copper Red
Image/Video URL: https://i.pinimg.com/originals/0d/b9/56/0db956e2c07e1580514b1c19587b4152.jpg
Likes: 2
Comments: 0
Post Type: image
User: ecemella
Date: 2023-05-24T22:32:51.000Z

--- Pin 2 ---
Success: True
Cost: $0.0010
Error: Bad input. Page does not exist. (code: dead_page)


## 1.2 Profiles - Collect profile data by URL

In [ ]:
# Collect profile data by URL
PROFILE_URLS = [
    "https://www.pinterest.com/boredpanda/",
    "https://www.pinterest.com/hadviser_com/",
]

print(f"Scraping {len(PROFILE_URLS)} profiles...")
for url in PROFILE_URLS:
    print(f"  - {url}")
print("This may take 1-3 minutes...\n")

async with client.scrape.pinterest.engine:
    results = await client.scrape.pinterest.profiles(url=PROFILE_URLS, timeout=240)

if isinstance(results, list):
    for i, result in enumerate(results):
        print(f"\n--- Profile {i+1} ---")
        print(f"Success: {result.success}")
        print(f"Cost: ${result.cost:.4f}" if result.cost else "Cost: N/A")
        if result.success and result.data:
            data = result.data
            print(f"Available keys: {list(data.keys()) if isinstance(data, dict) else 'N/A'}")
            if isinstance(data, dict):
                print(f"Username: {data.get('username', 'N/A')}")
                print(f"Full Name: {data.get('full_name', 'N/A')}")
                print(f"Followers: {data.get('followers', 'N/A')}")
                print(f"Following: {data.get('following', 'N/A')}")
                print(f"Bio: {str(data.get('biography', 'N/A'))[:100]}")
else:
    print(f"Success: {results.success}")
    print(f"Status: {results.status}")
    if results.success and results.data:
        print(f"Data: {str(results.data)[:300]}")
    else:
        print(f"Error: {results.error}")

## 1.3 Posts - Manual Trigger/Poll/Fetch

In [ ]:
# Manual trigger/poll/fetch for posts
PIN_URL = "https://www.pinterest.com/pin/3166662230556591/"

print(f"Triggering post collection: {PIN_URL}")

async with client.scrape.pinterest.engine:
    # Step 1: Trigger
    job = await client.scrape.pinterest.posts_trigger(url=PIN_URL)
    print(f"Snapshot ID: {job.snapshot_id}")

    # Step 2: Wait for completion
    print("Waiting for results...")
    await job.wait(timeout=240, poll_interval=10, verbose=True)

    # Step 3: Fetch results
    data = await job.fetch()
    print(f"\nFetched {len(data) if isinstance(data, list) else 1} record(s)")
    print(f"Data preview: {str(data)[:300]}")

---
# Part 2: PinterestSearchScraper (Discovery)

Discover Pinterest content using keywords and profile URLs.

## 2.1 Posts Discovery - by Keyword

In [ ]:
# Discover posts by keyword
KEYWORD = "spaghetti recipes"

print(f"Discovering posts for keyword: {KEYWORD}")
print("Using: type=discover_new, discover_by=keyword")
print("This may take 1-3 minutes...\n")

async with client.search.pinterest.engine:
    result = await client.search.pinterest.posts_by_keyword(
        keyword=KEYWORD,
        videos_only=False,
        timeout=240,
    )

print(f"Success: {result.success}")
print(f"Status: {result.status}")
print(f"Snapshot ID: {result.snapshot_id}")
print(f"Cost: ${result.cost:.4f}" if result.cost else "Cost: N/A")

if result.success and result.data:
    print("\n--- Discovered Posts ---")
    data = result.data
    if isinstance(data, list):
        valid = [p for p in data if isinstance(p, dict) and 'error' not in p]
        print(f"Number of posts discovered: {len(valid)}")
        if valid:
            print(f"Available keys: {list(valid[0].keys())}")
        for i, post in enumerate(valid[:5]):
            print(f"\nPost {i+1}:")
            print(f"  Title: {post.get('title', 'N/A')}")
            print(f"  URL: {post.get('url', 'N/A')}")
            print(f"  Image: {post.get('image_video_url', 'N/A')}")
            print(f"  Likes: {post.get('likes', 'N/A')}")
            print(f"  Comments: {post.get('comments_num', 'N/A')}")
            print(f"  Content: {str(post.get('content', 'N/A'))[:80]}")
    else:
        print(f"Data type: {type(data)}")
else:
    print(f"\nError: {result.error}")

## 2.2 Posts Discovery - by Keyword (videos only)

In [ ]:
# Discover video pins by keyword
KEYWORD = "workout routines"

print(f"Discovering video pins for: {KEYWORD}")
print("Filter: videos_only=True\n")

async with client.search.pinterest.engine:
    result = await client.search.pinterest.posts_by_keyword(
        keyword=KEYWORD,
        videos_only=True,
        timeout=240,
    )

print(f"Success: {result.success}")
print(f"Status: {result.status}")
print(f"Cost: ${result.cost:.4f}" if result.cost else "Cost: N/A")

if result.success and result.data:
    data = result.data
    if isinstance(data, list):
        valid = [p for p in data if isinstance(p, dict) and 'error' not in p]
        print(f"\nVideo pins found: {len(valid)}")
        for i, post in enumerate(valid[:3]):
            print(f"\nVideo {i+1}:")
            print(f"  Title: {post.get('title', 'N/A')}")
            print(f"  URL: {post.get('url', 'N/A')}")
else:
    print(f"\nError: {result.error}")

## 2.3 Posts Discovery - by Profile URL

In [ ]:
# Discover posts from a profile
PROFILE_URL = "https://www.pinterest.com/grandmapowpow/"

print(f"Discovering posts from profile: {PROFILE_URL}")
print("Using: type=discover_new, discover_by=profile_url")
print("This may take 1-3 minutes...\n")

async with client.search.pinterest.engine:
    result = await client.search.pinterest.posts_by_profile(
        url=PROFILE_URL,
        num_of_posts=10,
        timeout=240,
    )

print(f"Success: {result.success}")
print(f"Status: {result.status}")
print(f"Snapshot ID: {result.snapshot_id}")
print(f"Cost: ${result.cost:.4f}" if result.cost else "Cost: N/A")

if result.success and result.data:
    print("\n--- Posts from Profile ---")
    data = result.data
    if isinstance(data, list):
        valid = [p for p in data if isinstance(p, dict) and 'error' not in p]
        print(f"Number of posts: {len(valid)}")
        if valid:
            print(f"Available keys: {list(valid[0].keys())}")
        for i, post in enumerate(valid[:5]):
            print(f"\nPost {i+1}:")
            print(f"  Title: {post.get('title', 'N/A')}")
            print(f"  URL: {post.get('url', 'N/A')}")
            print(f"  Image: {post.get('image_video_url', 'N/A')}")
            print(f"  Likes: {post.get('likes', 'N/A')}")
    else:
        print(f"Data type: {type(data)}")
else:
    print(f"\nError: {result.error}")

## 2.4 Posts Discovery - by Profile URL (with date range)

In [ ]:
# Discover posts from a profile with date range filter
PROFILE_URL = "https://www.pinterest.com/grandmapowpow/"

print(f"Discovering posts from profile: {PROFILE_URL}")
print("Date range: 01-01-2025 to 03-01-2025\n")

async with client.search.pinterest.engine:
    result = await client.search.pinterest.posts_by_profile(
        url=PROFILE_URL,
        start_date="01-01-2025",
        end_date="03-01-2025",
        timeout=240,
    )

print(f"Success: {result.success}")
print(f"Status: {result.status}")
print(f"Cost: ${result.cost:.4f}" if result.cost else "Cost: N/A")

if result.success and result.data:
    data = result.data
    if isinstance(data, list):
        valid = [p for p in data if isinstance(p, dict) and 'error' not in p]
        print(f"\nPosts in date range: {len(valid)}")
        for i, post in enumerate(valid[:3]):
            print(f"\nPost {i+1}:")
            print(f"  Title: {post.get('title', 'N/A')}")
            print(f"  Date: {post.get('date', post.get('created_at', 'N/A'))}")
else:
    print(f"\nError: {result.error}")

## 2.5 Profiles Discovery - by Keyword

In [ ]:
# Discover profiles by keyword
KEYWORD = "mtv"

print(f"Discovering profiles for keyword: {KEYWORD}")
print("Using: type=discover_new, discover_by=keyword")
print("This may take 1-3 minutes...\n")

async with client.search.pinterest.engine:
    result = await client.search.pinterest.profiles(
        keyword=KEYWORD,
        timeout=240,
    )

print(f"Success: {result.success}")
print(f"Status: {result.status}")
print(f"Snapshot ID: {result.snapshot_id}")
print(f"Cost: ${result.cost:.4f}" if result.cost else "Cost: N/A")

if result.success and result.data:
    print("\n--- Discovered Profiles ---")
    data = result.data
    if isinstance(data, list):
        valid = [p for p in data if isinstance(p, dict) and 'error' not in p]
        print(f"Number of profiles: {len(valid)}")
        if valid:
            print(f"Available keys: {list(valid[0].keys())}")
        for i, profile in enumerate(valid[:5]):
            print(f"\nProfile {i+1}:")
            print(f"  Username: {profile.get('username', 'N/A')}")
            print(f"  Full Name: {profile.get('full_name', 'N/A')}")
            print(f"  Followers: {profile.get('followers', 'N/A')}")
            print(f"  URL: {profile.get('url', profile.get('profile_url', 'N/A'))}")
    else:
        print(f"Data type: {type(data)}")
else:
    print(f"\nError: {result.error}")

---
# Summary

## Part 1: PinterestScraper (URL-based)
- `client.scrape.pinterest.posts(url)` - Collect pin data by URL
- `client.scrape.pinterest.profiles(url)` - Collect profile data by URL
- `client.scrape.pinterest.posts_trigger(url)` - Manual trigger/poll/fetch

## Part 2: PinterestSearchScraper (Discovery)
- `client.search.pinterest.posts_by_keyword(keyword)` - Discover posts by keyword (`discover_by=keyword`)
- `client.search.pinterest.posts_by_profile(url)` - Discover posts from profile (`discover_by=profile_url`)
- `client.search.pinterest.profiles(keyword)` - Discover profiles by keyword (`discover_by=keyword`)

### Keyword Filters
- `videos_only=True` - Only video pins
- `new_posts=True` - Recent posts only
- `top_posts=True` - Top-performing posts
- `food=True` - Food-related content

### Profile URL Filters
- `num_of_posts=N` - Limit number of posts
- `start_date="MM-DD-YYYY"` - Filter by start date
- `end_date="MM-DD-YYYY"` - Filter by end date
- `posts_to_not_include=[...]` - Exclude specific post IDs